In [1]:
import os
from tqdm import tqdm
#
import sys
sys.path.append(
    os.path.abspath(
        os.path.join(
            os.getcwd(),
            '..',
            '..'
        ),
    )
)
from src.utils.json_operations import read_jsonl_file

In [2]:
TOKEN_MERGE_PATTERN = " |^| "

In [3]:
chembl_filepath = os.path.join("../../data","parsed-data","chembl_36_data.jsonl")
chembl_smiles = read_jsonl_file(chembl_filepath)

In [4]:
print(f"Number of SMILES in ChEMLB as of 2025-11-16: {len(chembl_smiles)}")

Number of SMILES in ChEMLB as of 2025-11-16: 2854815


In [5]:
chembl_smiles[0]

'Cc1cc(-c2csc(N=C(N)N)n2)cn1C'

In [6]:
data = []
for i in [["hug", 10], ["pug", 5], ["pun", 12], ["bun", 4], ["hugs", 5]]:
    data.extend([i[0]]*i[1])

In [7]:
def _pre_tokenize_with_hashtag_first_iteration(word:str=""):
    split_word = list(word.strip())
    hashtagged_tokens = split_word[:1] + list(map(lambda _char:'##'+_char, split_word[1:]))
    return hashtagged_tokens
#
def _pre_tokenize_with_hashtag_nth_iteration(word:str="", vocab_count:dict=dict()):
    hashtagged_tokens = []
    #
    _pointer = 0
    for outer_counter in range(len(word)):
        for inner_counter in range(len(word)-_pointer):
            max_match_subtoken = word[_pointer:len(word)-inner_counter]
            if outer_counter != 0:
                max_match_subtoken = "##" + max_match_subtoken
            if max_match_subtoken in vocab_count and _pointer<len(word):
                hashtagged_tokens.append(max_match_subtoken)
                _pointer +=len(max_match_subtoken.replace("##",""))
                break
    return hashtagged_tokens
#
def update_vocab_count(vocab_count:dict=dict(), hashtagged_tokens:list=list()):
    for _char in set(hashtagged_tokens):
        if _char not in vocab_count:
            vocab_count[_char] = 0
        vocab_count[_char] += hashtagged_tokens.count(_char)
    return vocab_count
#
def get_token_pair_count(token_pair_count:dict=dict(),hashtagged_tokens:list=list()):
    for i in range(len(hashtagged_tokens)-1):
        pair = f"{hashtagged_tokens[i]}{TOKEN_MERGE_PATTERN}{hashtagged_tokens[i+1]}"
        if pair not in token_pair_count:
            token_pair_count[pair] = 0
        token_pair_count[pair] +=1
    return token_pair_count
#
def get_merge_score(vocab_count:dict=dict(), token_pair_count:dict=dict()):
    token_pair_score = dict()
    for pair, count in token_pair_count.items():
        first_token, second_token = pair.split(TOKEN_MERGE_PATTERN)
        token_pair_score[pair] = token_pair_count[pair] / (vocab_count[first_token] * vocab_count[second_token])
    return token_pair_score
#
def get_token_with_max_score(merge_score:dict=dict()):
    max_pair = None
    max_score = -1
    for pair, score in merge_score.items():
        if score > max_score:
            max_score = score
            max_pair = pair
    return max_pair
#
def update_vocab_count_after_merge(vocab_count:dict=dict(), pair:str=''):
    if pair is None:
        print(f"Number of iterations given is exceeds the possible vocab space. Try decreasing the num_iterations parameter value")
        return vocab_count, True
    first_token, second_token = pair.split(TOKEN_MERGE_PATTERN)
    merged_token = first_token + second_token.replace("##","")
    #
    count_to_assign = min(vocab_count[first_token], vocab_count[second_token])
    vocab_count[merged_token] = count_to_assign
    # Should I reduce counts of the individual tokens? HF blog has not done that!
    vocab_count[first_token] -= count_to_assign
    vocab_count[second_token] -= count_to_assign
    return vocab_count, False
#
def reinitialize_vocab_count(vocab_keys:list=list()):
    vocab_count = dict()
    return vocab_count.fromkeys(vocab_keys,0)

In [8]:
def build_wordpiece(data:list=list(), num_iterations:int=5):
    vocab_count = dict()
    token_pair_count = dict()
    for word in data:
        hashtagged_tokens = _pre_tokenize_with_hashtag_first_iteration(word=word)
        vocab_count = update_vocab_count(vocab_count=vocab_count, hashtagged_tokens=hashtagged_tokens)
        token_pair_count = get_token_pair_count(token_pair_count=token_pair_count, hashtagged_tokens=hashtagged_tokens)
    #
    for iteration in tqdm(range(num_iterations)):
        merge_score = get_merge_score(vocab_count=vocab_count, token_pair_count=token_pair_count)
        pair_with_max_score = get_token_with_max_score(merge_score=merge_score)
        vocab_count, num_iteration_error = update_vocab_count_after_merge(vocab_count=vocab_count, pair=pair_with_max_score)
        if num_iteration_error:
            break
        #
        vocab_count = reinitialize_vocab_count(vocab_keys=list(vocab_count.keys()))
        token_pair_count = dict()
        for word in data:
            hashtagged_tokens = _pre_tokenize_with_hashtag_nth_iteration(word=word, vocab_count=vocab_count)
            vocab_count = update_vocab_count(vocab_count=vocab_count, hashtagged_tokens=hashtagged_tokens)
            token_pair_count = get_token_pair_count(token_pair_count=token_pair_count, hashtagged_tokens=hashtagged_tokens)
    return vocab_count

In [9]:
vocab_count = build_wordpiece(data=data, num_iterations=0)
vocab_count

0it [00:00, ?it/s]


{'h': 15, '##u': 36, '##g': 20, 'p': 17, '##n': 16, 'b': 4, '##s': 5}

In [10]:
vocab_count = build_wordpiece(data=data, num_iterations=1)
vocab_count

100%|██████████| 1/1 [00:00<00:00, 5924.16it/s]


{'h': 15,
 '##u': 36,
 '##g': 15,
 'p': 17,
 '##n': 16,
 'b': 4,
 '##s': 0,
 '##gs': 5}

In [11]:
vocab_count = build_wordpiece(data=data, num_iterations=2)
vocab_count

100%|██████████| 2/2 [00:00<00:00, 5797.24it/s]


{'h': 0,
 '##u': 21,
 '##g': 15,
 'p': 17,
 '##n': 16,
 'b': 4,
 '##s': 0,
 '##gs': 5,
 'hu': 15}

In [12]:
vocab_count = build_wordpiece(data=data, num_iterations=3)
vocab_count

100%|██████████| 3/3 [00:00<00:00, 10708.86it/s]


{'h': 0,
 '##u': 21,
 '##g': 15,
 'p': 17,
 '##n': 16,
 'b': 4,
 '##s': 0,
 '##gs': 0,
 'hu': 10,
 'hugs': 5}

In [13]:
vocab_count = build_wordpiece(data=data, num_iterations=9)
vocab_count

100%|██████████| 9/9 [00:00<00:00, 18044.33it/s]


{'h': 0,
 '##u': 0,
 '##g': 0,
 'p': 0,
 '##n': 0,
 'b': 0,
 '##s': 0,
 '##gs': 0,
 'hu': 0,
 'hugs': 5,
 'hug': 10,
 'pu': 0,
 'bu': 0,
 'bun': 4,
 'pug': 5,
 'pun': 12}

In [14]:
vocab_count = build_wordpiece(data=data, num_iterations=20)
vocab_count

 45%|████▌     | 9/20 [00:00<00:00, 4975.45it/s]

Number of iterations given is exceeds the possible vocab space. Try decreasing the num_iterations parameter value


{'h': 0,
 '##u': 0,
 '##g': 0,
 'p': 0,
 '##n': 0,
 'b': 0,
 '##s': 0,
 '##gs': 0,
 'hu': 0,
 'hugs': 5,
 'hug': 10,
 'pu': 0,
 'bu': 0,
 'bun': 4,
 'pug': 5,
 'pun': 12}

In [19]:
vocab_count = build_wordpiece(data=chembl_smiles[:100], num_iterations=20)

100%|██████████| 20/20 [00:07<00:00,  2.64it/s]


In [20]:
print(f"Total vocabulary size learned: {len(vocab_count)}")
largest_tokens = sorted(vocab_count.items(), key=lambda x:len(x[0]), reverse=True)
print(f"Larget tokens learned: {largest_tokens[:5]}")

Total vocabulary size learned: 59
Larget tokens learned: [('##Br-', 2), ('##no2', 1), ('##1no', 1), ('##3ns', 1), ('##Br', 0)]
